In [ ]:
#from Bio.PDB import *
#from Bio.PDB.DSSP import DSSP
#from Bio.PDB.DSSP import dssp_dict_from_pdb_file
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.lib.formats.libdcd import DCDFile

#faire 3 batch (1 de 401 à 600, 2 de 601 à 800, 3 de 801 à 1000)
dcd_file='batch{}.dcd'
psf_file='../../results/POPC-aa/POPC-{}/charmm-gui/namd/step5_input.psf'

coor_dir='../../results/POPC-aa/POPC-{}/analyses/traj{}/scripts/ring_coor/'
coor_files = coor_dir+'{}.dat'

In [ ]:
#Concatenate three 200ns blocks (1 : from 401 to 600, 2 : from 601 to 800, 3 : from 801 to 1000)
def preprocess(resname, traj_idx):
    !cat get_trajectory.sh | sed s=BBB={resname}= | sed s=TTT={traj_idx}= > temp.sh
    !bash temp.sh

In [13]:
#get coordinates
def get_coord(structure):
    # Create an empty list to store atom coordinates
    atom_coordinates = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    atom_coordinates.append(atom.get_coord())
    return atom_coordinates

def get_ca_coord(structure):
    # Create an empty list to store atom coordinates
    calpha_atoms = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                   if atom.get_name()=="CA":
                       calpha_atoms.append(atom) 
    CA_coordinates = [atom.get_coord() for atom in calpha_atoms]
    return CA_coordinates

#save new coordinates
def update(structure, modified_coordinates):
    # Update the atom coordinates in the structure:
    atom_index = 0

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    # Update the atom coordinates in the structure
                    atom.set_coord(modified_coordinates[atom_index])
                    atom_index += 1
    return structure

In [15]:
def regressi_vector(x, y, z):
    #determine a vector of the regression
    x1=np.column_stack((x,z))
    model = linear_model.LinearRegression()
    #print(x1, y)
    model.fit(x1,y)
    v1 = [model.coef_[0], 1, model.coef_[1]]
    return v1 / np.linalg.norm(v1) #vecteur unitaire
    #vecteur_directeur = np.array([reg.coef_[0], reg.coef_[1], reg.coef_[2]])

def angle_between(v1, v2):
    #v1 et v2 doivent être des vecteurs unitaires
    return np.arccos(np.clip(np.dot(v1, v2), -1.0, 1.0))


def angle_with_normal(structure, TM, v2):
    #calculate angle between the vector and the normal of the membrane
    x, y, z = get_ca_coord_part(structure, TM)
    v1=regressi_vector(x, y, z)
    return angle_between(v1, v2)

In [ ]:
# Version optimisée : extraction de tous les carbones en UN SEUL parcours de trajectoire
scs = ['SCW', 'SCF', 'SCY']

for sc in scs:
    trajs = [1, 2, 3]
    if sc == 'SCW':
        carbons = ['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
    else:
        carbons = ['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
    
    for traj_idx in trajs:
        preprocess(sc, traj_idx)
        for k in range(1, 4):
            u = mda.Universe(psf_file.format(sc), dcd_file.format(sc, traj_idx, k))
            print(f'\n=== {sc} traj{traj_idx} batch{k} ===')
            print(u)
        
            # Initialiser une liste par carbone (plus rapide que df.loc)
            data = {c: [] for c in carbons}
        
            # Sélection MDAnalysis pour TOUS les carbones du cycle
            all_carbons_str = " ".join(carbons)
        
            # UN SEUL PARCOURS de la trajectoire
            for ts in u.trajectory:
                fr = ts.frame
                if fr % 1000 == 0:
                    print(f'Frame {fr//1000}/20')
            
                # Sélectionner TOUS les atomes du cycle en une seule requête
                ring_atoms = u.select_atoms(f'resname {sc} and name {all_carbons_str}')
            
                # Trier les données par type de carbone
                for atom in ring_atoms:
                    c = atom.name
                    data[c].append([fr, c, atom.index, atom.resid, *atom.position])
        
            # Sauvegarder chaque carbone dans un fichier séparé
            for c in carbons:
                df = pd.DataFrame(data[c], columns=['#frame', 'C_type', 'index', 'resid', 'x', 'y', 'z'])
                output_file = coor_file.format(sc, traj_idx, f'{c}_coor_{k}')
                np.savetxt(
                    output_file,
                    df,
                    fmt='%i %s %i %i %.6f %.6f %.6f',
                    delimiter=' ',
                    newline='\n',
                    header='frame C_type index resid x y z',
                    comments='#'
                )
                print(f'  Saved {c}: {len(df)} entries (batch {k})')
    !rm trajectory*dcd
print("\nDone !!!")



=== SCYM traj1 batch1 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
  Saved CG: 43173 entries (batch 1)
  Saved CD1: 43173 entries (batch 1)
  Saved CD2: 43173 entries (batch 1)
  Saved CE1: 43173 entries (batch 1)
  Saved CE2: 43173 entries (batch 1)
  Saved CZ: 43173 entries (batch 1)

=== SCYM traj1 batch2 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
  Saved CG: 43175 entries (batch 2)
  Saved CD1: 43175 entries (batch 2)
  Saved CD2: 43175 entries (batch 2)
  Saved CE1: 43175 entries (batch 2)
  Saved CE2: 43175 entries (batch 2)
  Saved CZ: 43175 entries (batch 2)

=== SCYM traj1 batch3 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
Frame 2/20
  Saved CG: 43336 entries (batch 3)
  Saved CD1: 43336 entries (batch 3)
  Saved CD2: 43336 entries (batch 3)
  Saved CE1: 43336 entries (batch 3)
  Saved CE2: 43336 entries (batch 3)
  Saved CZ: 43336 entries (batch 3)

=== SCYM traj2 batch1 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
  Saved CG: 42812 entries (ba